In [13]:

# 评估facepp
import torch
from official_api.facepp import face_compare
from tqdm import tqdm
import os
import numpy as np

ths=[62.327,69.101,73.975]
result = {}
model_name="ArcFace"
with torch.no_grad():
    # 设置对抗样本目录路径 每月免费10000次调用，测三个刚好9000次，换个免费的继续测
    adv_samples_dirs = [
        f"data/FGSM_{model_name}_lfw_eps6_tpert4.4",
        f"data/MIM_{model_name}_lfw_eps6_tpert4.4",
        f"data/CW_{model_name}_lfw_eps16_tpert1",
        f"data/AdvMakeUP_lfw_tpert5.2",
        f"data/AT3D_{model_name}_eye_nose_lfw_eps5_tpert13.5",
        f"data/AdvFace_lfw_eps8_tpert5.7",
        f"data/TIPIM_{model_name}",
        f"data/SiblingAttack_{model_name}_lfw_eps0.15_tpert10.7",
        f"data/DiffAM",
        f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
    ]
    for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
        print("-----------------start white evaluate target method {0}-------------------\n".format(adv_samples_dir))
        save_path=f'apiresult/face++/{adv_samples_dir[5:]}.npy'
        if not os.path.exists(save_path):
            result[adv_samples_dir]= np.empty((0, 3), float)
            # 列出目录中的所有文件和文件夹
            all_files_and_dirs = os.listdir(adv_samples_dir)
            # 过滤出所有子文件夹
            subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
            for adv_pair in tqdm(subdirectories):
                base_dir = os.path.join(adv_samples_dir,adv_pair)
                adv_path = base_dir+"/adv.png"
                source_path = base_dir+"/source.png"
                target_path = base_dir+"/target.png"
                base_res = face_compare(face1_path=source_path, face2_path=target_path)
                FSS_res = face_compare(face1_path=adv_path, face2_path=source_path)
                FTS_res = face_compare(face1_path=adv_path, face2_path=target_path)
                if base_res is not None and FSS_res is not None and FTS_res is not None:
                    result[adv_samples_dir] = np.vstack([result[adv_samples_dir], [base_res,FSS_res,FTS_res]])
            print("face++在"+adv_samples_dir+"上失败了："+str(1000-len(result[adv_samples_dir])))
            np.save(save_path, np.array(result[adv_samples_dir]))
        else:
            result[adv_samples_dir] = np.load(save_path)
        num = len(result[adv_samples_dir])
        print("有效数据个数 ",num)
        for th in ths:
            print(th, np.sum(result[adv_samples_dir][:,0] > th)/num, np.sum(result[adv_samples_dir][:,2] > th)/num, np.sum((result[adv_samples_dir][:,1] > th) & (result[adv_samples_dir][:,2] > th))/num)

-----------------start white evaluate target method data/FGSM_ArcFace_lfw_eps6_tpert4.4-------------------

有效数据个数  974
62.327 0.018480492813141684 0.46919917864476385 0.46919917864476385
69.101 0.00513347022587269 0.2741273100616016 0.2741273100616016
73.975 0.002053388090349076 0.1375770020533881 0.13655030800821355
-----------------start white evaluate target method data/MIM_ArcFace_lfw_eps6_tpert4.4-------------------

有效数据个数  982
62.327 0.019348268839103868 0.774949083503055 0.769857433808554
69.101 0.006109979633401222 0.6720977596741344 0.6517311608961304
73.975 0.002036659877800407 0.5580448065173116 0.4959266802443992
-----------------start white evaluate target method data/CW_ArcFace_lfw_eps16_tpert1-------------------

有效数据个数  990
62.327 0.01919191919191919 0.09191919191919191 0.09191919191919191
69.101 0.006060606060606061 0.02727272727272727 0.02727272727272727
73.975 0.00202020202020202 0.006060606060606061 0.006060606060606061
-----------------start white evaluate target

In [14]:
print("face++ 0.01%FAR ASR1:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum(result[adv_samples_dir][:,2] > ths[1])/num*100:.1f}")
print("face++ 0.01%FAR ASR2:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum((result[adv_samples_dir][:, 1] > ths[1]) & (result[adv_samples_dir][:, 2] > ths[1])) / num*100:.1f}")

face++ 0.01%FAR ASR1:
27.4
67.2
2.7
5.6
59.0
60.0
57.2
92.0
14.1
89.8
face++ 0.01%FAR ASR2:
27.4
65.2
2.7
5.6
30.2
39.9
15.3
51.5
2.2
68.3


In [15]:
print("face++ 0.1%FAR ASR1:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum(result[adv_samples_dir][:,2] > ths[0])/num*100:.1f}")
print("face++ 0.1%FAR ASR2:")
for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
    num = len(result[adv_samples_dir])
    print(f"{np.sum((result[adv_samples_dir][:, 1] > ths[0]) & (result[adv_samples_dir][:, 2] > ths[0])) / num*100:.1f}")

face++ 0.1%FAR ASR1:
46.9
77.5
9.2
14.3
79.9
79.9
73.8
96.1
31.5
95.3
face++ 0.1%FAR ASR2:
46.9
77.0
9.2
14.3
55.1
66.3
38.6
72.5
11.8
84.4
